# Lab 12 — Mini Project: End-to-End Data Mining Pipeline

**Course:** Data Mining Lab  
**Notebook:** Lab 12 — Mini Project: End-to-End Data Mining Pipeline

## Objectives
- Select and load a real dataset (UCI Adult Income, with a labelled synthetic fallback only if download fails).
- Preprocess and explore data.
- Compare at least three classification models.
- Apply clustering and evaluate/interpret results.
- Present evidence-based findings and limitations.

> Run cells from top to bottom. The notebook uses a reproducible random seed. For Adult Income labs, it attempts to load the public UCI Adult dataset and uses a clearly labelled local synthetic fallback if the download is unavailable.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
COLUMNS = ["age","workclass","fnlwgt","education","education.num","marital.status",
           "occupation","relationship","race","sex","capital.gain","capital.loss",
           "hours.per.week","native.country","income"]

def load_adult():
    local = Path("adult.csv")
    if local.exists():
        data = pd.read_csv(local)
        if set(COLUMNS).issubset(data.columns):
            return data[COLUMNS].copy(), "local adult.csv"
    try:
        data = pd.read_csv(DATA_URL, names=COLUMNS, na_values="?", skipinitialspace=True)
        return data, "UCI Adult dataset (downloaded)"
    except Exception as exc:
        rng = np.random.default_rng(42)
        n = 1200
        age = rng.integers(18, 75, n)
        hours = np.clip(rng.normal(40, 12, n).round(), 1, 80).astype(int)
        edu = rng.integers(1, 16, n)
        gain = rng.choice([0,0,0,0,500,1500,5000,10000], n)
        score = -5 + 0.055*age + 0.10*edu + 0.025*hours + (gain>0)*1.0
        income = np.where(rng.random(n) < 1/(1+np.exp(-score)), ">50K", "<=50K")
        data = pd.DataFrame({
            "age":age,"workclass":rng.choice(["Private","Self-emp","Government"],n),
            "fnlwgt":rng.integers(20000,500000,n),
            "education":rng.choice(["HS-grad","Bachelors","Masters","Some-college"],n),
            "education.num":edu,"marital.status":rng.choice(["Married","Never-married","Divorced"],n),
            "occupation":rng.choice(["Tech","Sales","Service","Admin"],n),
            "relationship":rng.choice(["Husband","Not-in-family","Own-child"],n),
            "race":rng.choice(["White","Black","Other"],n),"sex":rng.choice(["Male","Female"],n),
            "capital.gain":gain,"capital.loss":rng.choice([0,0,0,500,1000],n),
            "hours.per.week":hours,"native.country":rng.choice(["United-States","Other"],n),
            "income":income})
        print("NOTICE: Public dataset unavailable; using reproducible synthetic teaching data. Reason:", str(exc)[:180])
        return data, "synthetic fallback (not UCI observations)"

df, DATA_SOURCE = load_adult()
print("Data source:", DATA_SOURCE, "| Shape:", df.shape)
display(df.head())

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Prepare target and features. Target is income category (>50K versus <=50K).
data=df.copy().replace("?",np.nan).drop_duplicates()
y=(data["income"].astype(str).str.strip().str.replace(".", "", regex=False)==">50K").astype(int)
X=data.drop(columns=["income"])
num=X.select_dtypes(include=np.number).columns.tolist()
cat=X.select_dtypes(exclude=np.number).columns.tolist()
numeric_pipe=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())])
categorical_pipe=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                           ("onehot",OneHotEncoder(handle_unknown="ignore"))])
preprocess=ColumnTransformer([("num",numeric_pipe,num),("cat",categorical_pipe,cat)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
print("Dataset:",DATA_SOURCE,"| rows:",len(data),"| features:",X.shape[1])

## 1. Preprocessing and exploratory analysis

In [ ]:
print("Missing values before pipeline:")
print(X.isna().sum())
display(data.describe(include="all").T.head(15))
data["income"].value_counts().plot(kind="bar",title="Income class distribution")
plt.ylabel("Records"); plt.show()

## 2. Classification model comparison

In [ ]:
models={
 "Decision Tree":DecisionTreeClassifier(max_depth=8,min_samples_leaf=10,random_state=42,class_weight="balanced"),
 "Random Forest":RandomForestClassifier(n_estimators=180,max_depth=14,min_samples_leaf=3,random_state=42,class_weight="balanced",n_jobs=-1),
 "Extra Trees":ExtraTreesClassifier(n_estimators=180,max_depth=14,min_samples_leaf=3,random_state=42,class_weight="balanced",n_jobs=-1)
}
results=[]
fitted={}
for name,estimator in models.items():
    pipe=Pipeline([("preprocess",preprocess),("model",estimator)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    results.append({"Model":name,"Accuracy":accuracy_score(y_test,pred),"F1-score":f1_score(y_test,pred)})
    fitted[name]=pipe
    print("\\n###",name)
    print(classification_report(y_test,pred,zero_division=0))
display(pd.DataFrame(results).sort_values("F1-score",ascending=False))

## 3. Confusion matrix for best F1 model

In [ ]:
best_name=max(results,key=lambda r:r["F1-score"])["Model"]
best_model=fitted[best_name]
print("Selected model:",best_name)
ConfusionMatrixDisplay.from_estimator(best_model,X_test,y_test)
plt.title(best_name+" — confusion matrix"); plt.show()

## 4. Clustering analysis

In [ ]:
# Unsupervised learning uses features only; income labels are not passed to K-Means.
X_cluster=preprocess.fit_transform(X)
# Limit rows for fast, readable clustering on ordinary student laptops.
rng=np.random.default_rng(42)
sample_idx=rng.choice(X_cluster.shape[0],size=min(2500,X_cluster.shape[0]),replace=False)
X_sample=X_cluster[sample_idx]
kmeans=KMeans(n_clusters=3,random_state=42,n_init=10)
cluster_labels=kmeans.fit_predict(X_sample)
print("Cluster sizes:",pd.Series(cluster_labels).value_counts().sort_index().to_dict())
projection=PCA(n_components=2,random_state=42)
coords=projection.fit_transform(X_sample.toarray() if hasattr(X_sample,"toarray") else X_sample)
plt.figure(figsize=(7,5))
plt.scatter(coords[:,0],coords[:,1],c=cluster_labels,s=8,alpha=.55)
plt.title("K-Means clusters (PCA projection)");plt.xlabel("PC1");plt.ylabel("PC2");plt.show()

## 5. Findings and limitations

### Report your observed results
- Record the selected dataset source and number of rows after duplicate removal.
- Compare the three classifiers using test accuracy and F1-score; explain why F1 may be useful for imbalanced classes.
- Describe the three cluster profiles using feature summaries. Do not assign real-world meaning to clusters without checking the underlying feature distributions.
- Mention that synthetic fallback data, if used, is only for demonstrating the workflow and must not be reported as real UCI findings.

### Limitations
The Adult dataset is historical and reflects its original collection context. Income prediction can encode social biases; results should not be used to make decisions about individuals. Model performance is not evidence of causation.